# Module 8: worked practice solutions

## Goal and route

Use the module's foundation, concepts and guided page first. This notebook supports the existing outcomes; it does not replace the complete study. This file contains answers: compare only after attempting the practice notebook.

Small check inputs are authored teaching examples. The final sandbox uses attributed real development records. Neither is a fresh final evaluation. Keep this notebook inside the complete source checkout/archive.


## Setup

Use the existing core environment with regression, visualization and verification extras for Modules 5–6; use the separate framework kernel for Module 8. No package installs or account tokens belong in this notebook.

In [ ]:
from pathlib import Path
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "2")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from practical_ml_ai.practice import check
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "curriculum/modules.json").is_file())
results = {}
plt.rcParams.update({"font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
print("Core environment" if 8 != 8 else "Framework environment with TensorFlow; CPU is sufficient")

## Task 1: dense_relu

Return np.maximum(0, x @ weights + bias). Here x is batch-by-input, weights is input-by-hidden and bias has one value per hidden unit. Write the operation yourself.

**Predict and explain:** Write each dimension before multiplying. One bias per hidden unit is shared across rows, not summed across the batch.

Write your prediction, attempted approach and what you changed here before running the check.

In [ ]:
def dense_relu(x, weights, bias):
    return np.maximum(0, np.asarray(x) @ np.asarray(weights) + np.asarray(bias))

In [ ]:
results["m08_dense"] = check("m08_dense", dense_relu)

## Task 2: select_checkpoint

Return a one-based epoch with minimum validation loss. Resolve ties with the earliest epoch. Training loss is supplied for interpretation, not selection.

**Predict and explain:** Predict the epoch from the illustrative loss table on the concepts page. Explain why the lowest training loss is not the rule.

Write your prediction, attempted approach and what you changed here before running the check.

In [ ]:
def select_checkpoint(train_loss, validation_loss):
    return validation_loss.index(min(validation_loss)) + 1

In [ ]:
results["m08_checkpoint"] = check("m08_checkpoint", select_checkpoint)

## Task 3: build_network

Return a built TensorFlow Sequential model: Input(shape=(inputs,)), Dense(width, activation="relu"), Dense(1, activation="sigmoid"). Do not train in this function.

**Predict and explain:** Predict the parameter count before constructing the model. Then change width and account for every new parameter.

Write your prediction, attempted approach and what you changed here before running the check.

In [ ]:
def build_network(inputs, width):
    import tensorflow as tf
    return tf.keras.Sequential([tf.keras.Input(shape=(inputs,)), tf.keras.layers.Dense(width, activation="relu"), tf.keras.layers.Dense(1, activation="sigmoid")])

In [ ]:
results["m08_network"] = check("m08_network", build_network)

## Apply the functions to real development records

This supplied scaffold uses your checked functions. If a check needs work, return to its function and rerun from the setup cell. A green local check supports these cases, not every possible implementation or your explanation.

Before running, write which inputs and labels are available and which partition may select a decision. Compare the exact tables with the figure; no chart-reading requirement substitutes for the numeric evidence.

In [ ]:
demo_verified = False
if len(results) == 3 and all(item["state"] == "passed" for item in results.values()):
    import tensorflow as tf
    from sklearn.preprocessing import StandardScaler
    from practical_ml_ai.continuation_data import wine_splits
    import tempfile
    split, features, provenance = wine_splits()
    # A small, explicitly bounded development slice for learning mechanics, not a benchmark.
    training, validation = split["train"].iloc[:256], split["validation"].iloc[:64]
    scaler = StandardScaler().fit(training[features])
    x_train = scaler.transform(training[features]).astype("float32")
    x_validation = scaler.transform(validation[features]).astype("float32")
    y_train = training.label.to_numpy(dtype="float32").reshape(-1, 1)
    y_validation = validation.label.to_numpy(dtype="float32").reshape(-1, 1)
    tf.keras.utils.set_random_seed(42)
    model = build_network(len(features), 8)
    optimizer = tf.keras.optimizers.Adam(learning_rate=.005)
    loss_function = tf.keras.losses.BinaryCrossentropy()
    history, checkpoints = [], []
    for epoch in range(1, 9):
        for start in range(0, len(training), 32):
            with tf.GradientTape() as tape:
                loss = loss_function(y_train[start:start+32], model(x_train[start:start+32], training=True))
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        history.append({"epoch": epoch, "train_loss": float(loss_function(y_train, model(x_train, training=False))),
                        "validation_loss": float(loss_function(y_validation, model(x_validation, training=False)))})
        checkpoints.append([weight.copy() for weight in model.get_weights()])
    history = pd.DataFrame(history)
    chosen = select_checkpoint(history.train_loss.tolist(), history.validation_loss.tolist())
    model.set_weights(checkpoints[chosen-1])
    ROOT.joinpath("runs").mkdir(exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix="practice08-", dir=ROOT / "runs"))
    model.save(run_dir / "model.keras")
    reloaded = tf.keras.models.load_model(run_dir / "model.keras", safe_mode=True)
    np.testing.assert_allclose(model(x_validation, training=False), reloaded(x_validation, training=False), atol=1e-6)
    display(history.round(4))
    ax = history.set_index("epoch").plot(figsize=(8, 4), style=["o-", "s--"], color=["#17658b", "#a65c00"])
    ax.axvline(chosen, color="#303030", linestyle=":", label="Selected epoch")
    ax.set(title="Bounded development training; selection uses validation loss", xlabel="Epoch", ylabel="Binary cross-entropy")
    ax.legend(); plt.tight_layout(); plt.show()
    demo_verified = len(history) == 8 and model.count_params() == len(features)*8+8+8+1
    print("Source: UCI Wine Quality, DOI 10.24432/C56S3T. Training/validation slices:", len(training), len(validation))
    print("Selected epoch:", chosen, "Saved model:", run_dir / "model.keras")
    print("Replay checks serialization. The complete Module 8 study separately compares held-out errors with its logistic baseline.")
else:
    print("Development sandbox not run: finish the three functions and rerun from Setup.")

## Independent transfer and handoff

Return to the module practice page for the changed IT task, diagnosis and decision memo. Retain your code, predictions, feedback, one manually checked number and a short explanation of a failure. Then run the complete module walkthrough for its full baseline/final-evaluation/artifact contract.

An unattempted starter executes with visible `not_attempted` feedback and skips the sandbox; that is not completed learning. Solutions demonstrate a worked approach, not evidence that a learner can reproduce it independently. Human piloting is pending.
